# J4 - PySpark Avancé : S3, Window Functions, Broadcast Join, Delta analytique
## Stack : PySpark 3.5, Delta Lake, boto3, Databricks Free Edition
## Données : SIRENE Loire-Atlantique - 420 411 lignes brutes → 134 661 actifs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
import boto3

VOLUME_PATH = "/Volumes/workspace/default/raw_data"
DELTA_CLEAN = f"{VOLUME_PATH}/sirene_clean_delta"
DELTA_ANALYTIQUE = f"{VOLUME_PATH}/sirene_analytique_delta"

In [0]:
# ── Lecture CSV brut ────────────────────────────────────────────────
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{VOLUME_PATH}/sirene/data.csv")
)

# ── Renommage 24 colonnes utiles ─────────────────────────────────────
rename_map = {
    "SIREN": "siren", "NIC": "nic", "SIRET": "siret",
    "Statut de diffusion de l'établissement": "statut_diffusion",
    "Date de création de l'établissement": "date_creation_etab",
    "Tranche de l'effectif de l'établissement": "tranche_effectif",
    "Activité principale de l'établissement8": "activite_principale_etab",
    "Etablissement siège": "etablissement_siege",
    "Code postal de l'établissement": "code_postal",
    "Commune de l'établissement": "commune",
    "Code commune de l'établissement": "code_commune",
    "Code du département de l'établissement": "code_departement",
    "Département de l'établissement": "departement",
    "Code de la région de l'établissement": "code_region",
    "Région de l'établissement": "region",
    "Etat administratif de l'établissement": "etat_admin_etab",
    "Date de fermeture de l'établissement": "date_fermeture_etab",
    "Dénomination de l'unité légale": "denomination_unite_legale",
    "Catégorie de l'entreprise": "categorie_entreprise",
    "Etat administratif de l'unité légale": "etat_admin_ul",
    "Caractère employeur de l'unité légale": "caractere_employeur",
    "Activité principale de l'unité légale": "activite_principale_ul",
    "Catégorie juridique de l'unité légale": "categorie_juridique",
    "Date de création de l'unité légale": "date_creation_ul",
}
df_renamed = df_raw
for old, new in rename_map.items():
    if old in df_renamed.columns:
        df_renamed = df_renamed.withColumnRenamed(old, new)
df_renamed = df_renamed.select(list(rename_map.values()))

# ── clean_nd + filtres RGPD + colonnes dérivées ──────────────────────
for c in ["denomination_unite_legale", "activite_principale_etab", "categorie_entreprise"]:
    df_renamed = df_renamed.withColumn(c,
        F.when((F.col(c) == "[ND]") | (F.col(c) == ""), None).otherwise(F.col(c)))

df_clean = (
    df_renamed
    .filter(F.col("etat_admin_etab") == "Actif")
    .filter(F.col("statut_diffusion") != "P")
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn("siret_calcule", F.concat(
        F.lpad(F.col("siren").cast("string"), 9, "0"),
        F.lpad(F.col("nic").cast("string"),  5, "0")))
    .withColumn("est_siege",
        F.when(F.col("etablissement_siege").isin("oui","true","True","1"), True)
         .when(F.col("etablissement_siege").isin("non","false","False","0"), False)
         .otherwise(None))
)

cnt = df_clean.count()
print(f"df_clean : {cnt:,} lignes - {len(df_clean.columns)} colonnes")
assert cnt == 134661, f"Count inattendu : {cnt} - vérifier filtres"

df_clean : 134,661 lignes - 27 colonnes


In [0]:
# Récupérer les credentials depuis Databricks Secrets
access_key = dbutils.secrets.get(scope="aws-credentials", key="aws-access-key")
secret_key = dbutils.secrets.get(scope="aws-credentials", key="aws-secret-key")

s3 = boto3.client(
    's3',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='eu-west-1'
)

print("Client boto3 S3 initialisé")

Client boto3 S3 initialisé


In [0]:
# Lister les objets dans le dossier raw/sirene/
response = s3.list_objects_v2(
    Bucket='alan-data-lake-fr',
    Prefix='raw/sirene/'
)

print("Fichiers dans s3://alan-data-lake-fr/raw/sirene/ :")
for obj in response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    modified = obj['LastModified'].strftime('%Y-%m-%d %H:%M')
    print(f"  {obj['Key']:<55} {size_kb:>8.0f} KB  {modified}")

# Valider que la clé exacte utilisée par le S3KeySensor est présente
target_key = "raw/sirene/annee=2024/mois=01/data.csv"
try:
    meta = s3.head_object(Bucket='alan-data-lake-fr', Key=target_key)
    print(f"\nClé S3KeySensor présente : {target_key}")
    print(f"Taille : {meta['ContentLength']:,} octets")
except Exception as e:
    print(f"\nClé introuvable : {target_key}")

Fichiers dans s3://alan-data-lake-fr/raw/sirene/ :
  raw/sirene/annee=2024/mois=01/                                 0 KB  2026-05-04 17:56
  raw/sirene/annee=2024/mois=01/data.csv                    410541 KB  2026-05-04 18:10
  raw/sirene/annee=2024/mois=02/                                 0 KB  2026-05-04 17:56
  raw/sirene/annee=2024/mois=03/                                 0 KB  2026-05-04 17:56
  raw/sirene/test_snowpipe_20260515_141648.parquet              16 KB  2026-05-15 18:28
  raw/sirene/test_snowpipe_20260515_143209.parquet              16 KB  2026-05-15 18:32
  raw/sirene/test_snowpipe_20260515_143558.parquet              16 KB  2026-05-15 18:36
  raw/sirene/test_snowpipe_20260515_144056.parquet              16 KB  2026-05-15 18:43
  raw/sirene/test_snowpipe_20260515_144552.parquet              16 KB  2026-05-15 18:45
  raw/sirene/test_snowpipe_20260515_144944.parquet              16 KB  2026-05-15 18:49

Clé S3KeySensor présente : raw/sirene/annee=2024/mois=01/data.csv
Ta

In [0]:
try:
    spark.conf.set("fs.s3a.access.key",    access_key)
    spark.conf.set("fs.s3a.secret.key",    secret_key)
    spark.conf.set("fs.s3a.endpoint.region", "eu-west-1")

    df_s3 = (spark.read
        .option("header", "true")
        .option("sep", ";")
        .csv("s3a://alan-data-lake-fr/raw/sirene/annee=2024/mois=01/data.csv")
        .limit(5))
    df_s3.show(2, truncate=40)
    print("✅ Lecture Spark s3a:// opérationnelle")

except Exception as e:
    print(f"s3a non disponible en serverless Azure : {type(e).__name__}")
    print("→ Normal. boto3 (Cell 5) couvre l'accès S3 pour ce notebook.")
    print("→ En prod : External Location Unity Catalog pour accès cross-cloud.")

s3a non disponible en serverless Azure : AnalysisException
→ Normal. boto3 (Cell 5) couvre l'accès S3 pour ce notebook.
→ En prod : External Location Unity Catalog pour accès cross-cloud.


In [0]:
df_stats = (
    df_clean
    .groupBy("code_departement", "commune", "code_commune")
    .agg(
        F.count("siret").alias("nb_etablissements"),
        F.countDistinct("activite_principale_etab").alias("nb_activites_distinctes"),
        F.sum(F.when(F.col("est_siege") == True, 1).otherwise(0)).alias("nb_sieges"),
        F.sum(F.when(F.col("categorie_entreprise") == "PME", 1).otherwise(0)).alias("nb_pme"),
        F.sum(F.when(F.col("categorie_entreprise") == "GE",  1).otherwise(0)).alias("nb_ge"),
        F.sum(F.when(F.col("categorie_entreprise") == "ETI", 1).otherwise(0)).alias("nb_eti"),
    )
    .withColumn("pct_pme",
        F.round(F.col("nb_pme") / F.col("nb_etablissements") * 100, 1))
)

print(f"Communes analysées : {df_stats.count():,}")
display(df_stats.orderBy(F.desc("nb_etablissements")).limit(10))

Communes analysées : 34


code_departement,commune,code_commune,nb_etablissements,nb_activites_distinctes,nb_sieges,nb_pme,nb_ge,nb_eti,pct_pme
44,NANTES,44109,74724,245,65286,43275,1709,2134,57.9
44,SAINT-HERBLAIN,44162,10268,144,8471,5969,474,1011,58.1
44,REZE,44143,7159,119,6267,4480,158,270,62.6
44,VERTOU,44215,4819,104,4324,2919,122,154,60.6
44,ORVAULT,44114,4742,93,4146,2706,173,220,57.1
44,CARQUEFOU,44026,4253,102,3566,2407,170,244,56.6
44,SAINT-SEBASTIEN-SUR-LOIRE,44190,3674,89,3300,2211,53,88,60.2
44,BOUGUENAIS,44020,3258,101,2818,1981,104,167,60.8
44,LA CHAPELLE-SUR-ERDRE,44035,3098,87,2786,1906,76,98,61.5
44,COUERON,44047,3040,105,2706,1821,57,160,59.9


In [0]:
# Window 1 : rang au sein du département (descendant par nb_etablissements)
w_rank = (
    Window
    .partitionBy("code_departement")
    .orderBy(F.desc("nb_etablissements"))
)

# Window 2 : cumul (somme courante du début jusqu'à la ligne actuelle)
w_cumul = (
    Window
    .partitionBy("code_departement")
    .orderBy(F.desc("nb_etablissements"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Window 3 : total du département (toutes lignes, pour le %)
# partitionBy sans orderBy = fenêtre sur tout le département
w_total = Window.partitionBy("code_departement")

print("3 WindowSpecs définies")
print("w_rank  → RANK() dans le département")
print("w_cumul → SUM() cumulé depuis le début")
print("w_total → SUM() total du département (dénominateur)")

3 WindowSpecs définies
w_rank  → RANK() dans le département
w_cumul → SUM() cumulé depuis le début
w_total → SUM() total du département (dénominateur)


In [0]:
df_ranked = (
    df_stats
    # rank() : sauts en cas d'ex-aequo (ex: 1,2,2,4)
    .withColumn("rang",F.rank().over(w_rank))
    # dense_rank() : pas de saut (ex: 1,2,2,3)
    .withColumn("dense_rang", F.dense_rank().over(w_rank))
    # Cumul d'établissements (top N communes couvrent X établissements)
    .withColumn("cumul_etab", F.sum("nb_etablissements").over(w_cumul))
    # Total département (dénominateur pour le %)
    .withColumn("total_dep",  F.sum("nb_etablissements").over(w_total))
    # % du département représenté par cette commune
    .withColumn("pct_dep",
        F.round(F.col("nb_etablissements") / F.col("total_dep") * 100, 2))
    # % cumulé (top 1 commune = X%, top 2 = Y%, ...)
    .withColumn("pct_cumul",
        F.round(F.col("cumul_etab") / F.col("total_dep") * 100, 1))
)

# Top 10 - Nantes devrait être rang 1
display(
    df_ranked
    .select("commune","nb_etablissements","rang","dense_rang",
            "cumul_etab","pct_dep","pct_cumul")
    .orderBy("rang")
    .limit(10)
)

commune,nb_etablissements,rang,dense_rang,cumul_etab,pct_dep,pct_cumul
NANTES,74724,1,1,74724,55.49,55.5
SAINT-HERBLAIN,10268,2,2,84992,7.63,63.1
REZE,7159,3,3,92151,5.32,68.4
VERTOU,4819,4,4,96970,3.58,72.0
ORVAULT,4742,5,5,101712,3.52,75.5
CARQUEFOU,4253,6,6,105965,3.16,78.7
SAINT-SEBASTIEN-SUR-LOIRE,3674,7,7,109639,2.73,81.4
BOUGUENAIS,3258,8,8,112897,2.42,83.8
LA CHAPELLE-SUR-ERDRE,3098,9,9,115995,2.3,86.1
COUERON,3040,10,10,119035,2.26,88.4


In [0]:
# Voir le plan d'exécution
df_ranked.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   PhotonResultStage (24)
   +- PhotonColumnarToRow (23)
      +- PhotonProject (22)
         +- PhotonWindow (21)
            +- PhotonWindow (20)
               +- PhotonSort (19)
                  +- PhotonShuffleExchangeSource (18)
                     +- PhotonShuffleMapStage (17)
                        +- PhotonShuffleExchangeSink (16)
                           +- PhotonProject (15)
                              +- PhotonGroupingAgg (14)
                                 +- PhotonShuffleExchangeSource (13)
                                    +- PhotonShuffleMapStage (12)
                                       +- PhotonShuffleExchangeSink (11)
                                          +- PhotonGroupingAgg (10)
                                             +- PhotonGroupingAgg (9)
                                                +- PhotonShuffleExchangeSource (8)
                                                   +- Ph

In [0]:
naf_ref = [
    ("01", "Agriculture, chasse et services annexes"),
    ("02", "Sylviculture et exploitation forestière"),
    ("03", "Pêche et aquaculture"),
    ("05", "Extraction de houille et de lignite"),
    ("06", "Extraction d'hydrocarbures"),
    ("07", "Extraction de minerais métalliques"),
    ("08", "Autres industries extractives"),
    ("09", "Services de soutien aux industries extractives"),
    ("10", "Industries alimentaires"),
    ("11", "Fabrication de boissons"),
    ("12", "Fabrication de produits à base de tabac"),
    ("13", "Fabrication de textiles"),
    ("14", "Industrie de l'habillement"),
    ("15", "Industrie du cuir et de la chaussure"),
    ("16", "Travail du bois, industries du papier"),
    ("17", "Industrie du papier et du carton"),
    ("18", "Imprimerie et reproduction d'enregistrements"),
    ("19", "Cokéfaction et raffinage"),
    ("20", "Industrie chimique"),
    ("21", "Industrie pharmaceutique"),
    ("22", "Fabrication de produits en caoutchouc et en plastique"),
    ("23", "Fabrication d'autres produits minéraux non métalliques"),
    ("24", "Métallurgie"),
    ("25", "Fabrication de produits métalliques (hors machines)"),
    ("26", "Fabrication de produits informatiques, électroniques et optiques"),
    ("27", "Fabrication d'équipements électriques"),
    ("28", "Fabrication de machines et équipements n.c.a."),
    ("29", "Industrie automobile"),
    ("30", "Fabrication d'autres matériels de transport"),
    ("31", "Fabrication de meubles"),
    ("32", "Autres industries manufacturières"),
    ("33", "Réparation et installation de machines et d'équipements"),
    ("35", "Production et distribution d'électricité, de gaz, de vapeur"),
    ("36", "Captage, traitement et distribution d'eau"),
    ("37", "Collecte et traitement des eaux usées"),
    ("38", "Collecte, traitement et élimination des déchets"),
    ("39", "Dépollution et autres services de gestion des déchets"),
    ("41", "Construction de bâtiments"),
    ("42", "Génie civil"),
    ("43", "Travaux de construction spécialisés"),
    ("45", "Commerce et réparation automobiles"),
    ("46", "Commerce de gros"),
    ("47", "Commerce de détail"),
    ("49", "Transports terrestres et transport par conduites"),
    ("50", "Transports par eau"),
    ("51", "Transports aériens"),
    ("52", "Entreposage et services auxiliaires des transports"),
    ("53", "Activités de poste et de courrier"),
    ("55", "Hébergement"),
    ("56", "Restauration"),
    ("58", "Édition"),
    ("59", "Production de films, vidéo et programmes de télévision"),
    ("60", "Programmation et diffusion"),
    ("61", "Télécommunications"),
    ("62", "Activités informatiques"),
    ("63", "Services d'information"),
    ("64", "Services financiers"),
    ("65", "Assurance"),
    ("66", "Activités auxiliaires de services financiers et d'assurance"),
    ("68", "Activités immobilières"),
    ("69", "Activités juridiques et comptables"),
    ("70", "Activités des sièges sociaux ; conseil de gestion"),
    ("71", "Activités d'architecture et d'ingénierie"),
    ("72", "Recherche-développement scientifique"),
    ("73", "Publicité et études de marché"),
    ("74", "Autres activités spécialisées, scientifiques et techniques"),
    ("75", "Activités vétérinaires"),
    ("77", "Activités de location et location-bail"),
    ("78", "Activités liées à l'emploi"),
    ("79", "Agences de voyage, voyagistes, réservation"),
    ("80", "Enquêtes et sécurité"),
    ("81", "Services relatifs aux bâtiments et aménagement paysager"),
    ("82", "Activités administratives et de soutien aux entreprises"),
    ("84", "Administration publique"),
    ("85", "Enseignement"),
    ("86", "Santé humaine"),
    ("87", "Hébergement médico-social et social"),
    ("88", "Action sociale sans hébergement"),
    ("90", "Activités créatives, artistiques et de spectacle"),
    ("91", "Bibliothèques, archives, musées et autres activités culturelles"),
    ("92", "Organisation de jeux de hasard et d'argent"),
    ("93", "Activités sportives, récréatives et de loisirs"),
    ("94", "Activités des organisations associatives"),
    ("95", "Réparation d'ordinateurs et de biens personnels"),
    ("96", "Autres services personnels"),
    ("97", "Activités des ménages employeurs de personnel domestique"),
    ("98", "Activités indifférenciées des ménages producteurs"),
    ("99", "Activités des organisations extraterritoriales"),
]
df_naf = spark.createDataFrame(naf_ref, ["code_naf_2", "libelle_secteur"])
print(f"Table NAF : {df_naf.count()} secteurs")
df_naf.show(truncate=False)

Table NAF : 88 secteurs
+----------+----------------------------------------------+
|code_naf_2|libelle_secteur                               |
+----------+----------------------------------------------+
|01        |Agriculture, chasse et services annexes       |
|02        |Sylviculture et exploitation forestière       |
|03        |Pêche et aquaculture                          |
|05        |Extraction de houille et de lignite           |
|06        |Extraction d'hydrocarbures                    |
|07        |Extraction de minerais métalliques            |
|08        |Autres industries extractives                 |
|09        |Services de soutien aux industries extractives|
|10        |Industries alimentaires                       |
|11        |Fabrication de boissons                       |
|12        |Fabrication de produits à base de tabac       |
|13        |Fabrication de textiles                       |
|14        |Industrie de l'habillement                    |
|15        |Indu

In [0]:
# Cell 11 bis - Diagnostiquer les activite_principale_etab manquants
df_clean_naf.filter(F.col("activite_principale_etab").isNull() | (F.col("activite_principale_etab") == "")).count()

125652

In [0]:
# Extraire les 2 premiers caractères du code NAF (ex: "47.11A" → "47")
df_clean_naf = df_clean.withColumn(
    "code_naf_2",
    F.substring(F.col("activite_principale_etab"), 1, 2)  # 1-indexé en PySpark
)

# Broadcast join - F.broadcast() force l'envoi de df_naf (petite table) à tous les nœuds
# Évite un shuffle (SortMergeJoin) sur la grande table df_clean
df_enrichi = (
    df_clean_naf
    .join(F.broadcast(df_naf), on="code_naf_2", how="left")
)

# Vérifier : left join ne doit pas perdre de lignes
cnt_enrichi = df_enrichi.count()
print(f"df_enrichi : {cnt_enrichi:,} lignes")
assert cnt_enrichi == 134661, f"Le left join a perdu des lignes : {cnt_enrichi}"
print("Count préservé - left join correct")

df_enrichi : 134,661 lignes
Count préservé - left join correct


In [0]:
# Voir le plan physique - chercher "BroadcastHashJoin" dans l'output
df_enrichi.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonProject (12)
         +- PhotonBroadcastHashJoin LeftOuter (11)
            :- PhotonProject (5)
            :  +- PhotonProject (4)
            :     +- PhotonFilter (3)
            :        +- PhotonRowToColumnar (2)
            :           +- Scan csv  (1)
            +- PhotonShuffleExchangeSource (10)
               +- PhotonShuffleMapStage (9)
                  +- PhotonShuffleExchangeSink (8)
                     +- PhotonRowToColumnar (7)
                        +- LocalTableScan (6)


(1) Scan csv 
Output [24]: [SIREN#21655, NIC#21656, SIRET#21657L, Statut de diffusion de l'établissement#21658, Date de création de l'établissement#21659, Tranche de l'effectif de l'établissement#21660, Activité principale de l'établissement8#21663, Etablissement siège#21665, Code postal de l'établissement#21672, Commune de l'établissement#21673, Code commune de

In [0]:
# Comptage par secteur + % du total
total_etab = 134661  # valeur fixe - évite une action supplémentaire

df_secteurs = (
    df_enrichi
    .groupBy("libelle_secteur")
    .agg(F.count("siret").alias("nb_etablissements"))
    .withColumn("pct",
        F.round(F.col("nb_etablissements") / total_etab * 100, 1))
    .orderBy(F.desc("nb_etablissements"))
)

# Afficher (libelle_secteur NULL = code NAF absent de la référence → secteurs mineurs)
display(df_secteurs)

libelle_secteur,nb_etablissements,pct
null,125652,93.3
Travaux de construction spécialisés,3234,2.4
Autres services personnels,923,0.7
Services relatifs aux bâtiments et aménagement paysager,622,0.5
Restauration,526,0.4
Transports terrestres et transport par conduites,465,0.3
Industries alimentaires,395,0.3
Commerce et réparation automobiles,356,0.3
Réparation d'ordinateurs et de biens personnels,308,0.2
Autres industries manufacturières,298,0.2


In [0]:
df_analytique = (
    df_enrichi
    .groupBy("code_departement", "commune", "code_commune", "libelle_secteur")
    .agg(
        F.count("siret")                            .alias("nb_etablissements"),
        F.countDistinct("siren")                   .alias("nb_entreprises_uniques"),
        F.sum(F.when(F.col("est_siege")==True,1).otherwise(0)).alias("nb_sieges"),
        F.sum(F.when(F.col("categorie_entreprise")=="PME",1).otherwise(0)).alias("nb_pme"),
        F.sum(F.when(F.col("categorie_entreprise")=="GE", 1).otherwise(0)).alias("nb_ge"),
        F.sum(F.when(F.col("categorie_entreprise")=="ETI",1).otherwise(0)).alias("nb_eti"),
        F.max("loaded_at")                          .alias("loaded_at"),
    )
)

cnt_ana = df_analytique.count()
print(f"df_analytique : {cnt_ana:,} lignes (combinaisons commune × secteur)")
display(df_analytique.orderBy(F.desc("nb_etablissements")).limit(10))

df_analytique : 680 lignes (combinaisons commune × secteur)


code_departement,commune,code_commune,libelle_secteur,nb_etablissements,nb_entreprises_uniques,nb_sieges,nb_pme,nb_ge,nb_eti,loaded_at
44,NANTES,44109,null,71086,67007,61848,39674,1708,2116,2026-07-08T14:10:40.275Z
44,SAINT-HERBLAIN,44162,null,9436,9108,7688,5159,472,997,2026-07-08T14:10:40.275Z
44,REZE,44143,null,6495,6276,5637,3831,156,263,2026-07-08T14:10:40.275Z
44,VERTOU,44215,null,4457,4321,3984,2564,121,150,2026-07-08T14:10:40.275Z
44,ORVAULT,44114,null,4413,4297,3836,2380,173,217,2026-07-08T14:10:40.275Z
44,CARQUEFOU,44026,null,3954,3850,3288,2112,168,243,2026-07-08T14:10:40.275Z
44,SAINT-SEBASTIEN-SUR-LOIRE,44190,null,3395,3320,3041,1933,53,88,2026-07-08T14:10:40.275Z
44,LA CHAPELLE-SUR-ERDRE,44035,null,2896,2835,2596,1709,75,95,2026-07-08T14:10:40.275Z
44,BOUGUENAIS,44020,null,2832,2728,2411,1563,104,162,2026-07-08T14:10:40.275Z
44,COUERON,44047,null,2656,2584,2339,1442,55,158,2026-07-08T14:10:40.275Z


In [0]:
(
    df_analytique
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # nécessaire si le schéma a changé
    .partitionBy("code_departement")
    .save(DELTA_ANALYTIQUE)
)
print(f"Table Delta écrite dans {DELTA_ANALYTIQUE}")

Table Delta écrite dans /Volumes/workspace/default/raw_data/sirene_analytique_delta


In [0]:
dt_ana = DeltaTable.forPath(spark, DELTA_ANALYTIQUE)

# Historique des versions
print("=== Historique Delta : sirene_analytique_delta ===")
dt_ana.history().select("version", "timestamp", "operation", "operationMetrics").show(5, truncate=60)

# Count final
count_final = spark.read.format("delta").load(DELTA_ANALYTIQUE).count()
print(f"\nsirene_analytique_delta : {count_final:,} lignes")

# Inspecter la structure physique dans Unity Catalog Volumes
print("\n=== Contenu du Volume ===")
for f in dbutils.fs.ls(DELTA_ANALYTIQUE):
    print(f"{f.name:<45} {f.size:>10,}")

=== Historique Delta : sirene_analytique_delta ===
+-------+-------------------+---------+------------------------------------------------------------+
|version|          timestamp|operation|                                            operationMetrics|
+-------+-------------------+---------+------------------------------------------------------------+
|      1|2026-07-08 14:10:49|    WRITE|{numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> ...|
|      0|2026-07-08 14:03:26|    WRITE|{numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> ...|
+-------+-------------------+---------+------------------------------------------------------------+


sirene_analytique_delta : 680 lignes

=== Contenu du Volume ===
_delta_log/                                            0
code_departement=44/                                   0


In [0]:
df_ranked.createOrReplaceTempView("v_communes_ranked")
df_analytique.createOrReplaceTempView("v_sirene_analytique")

print("Vues temporaires créées :")
for v in ["v_communes_ranked", "v_sirene_analytique"]:
    cnt = spark.sql(f"SELECT COUNT(*) FROM {v}").collect()[0][0]
    print(f"{v} → {cnt:,} lignes")

Vues temporaires créées :
v_communes_ranked → 34 lignes
v_sirene_analytique → 680 lignes


In [0]:
%sql
-- Top 10 communes par nombre d'établissements actifs
SELECT
    commune,
    nb_etablissements,
    rang,
    dense_rang,
    nb_activites_distinctes,
    cumul_etab,
    pct_dep                    AS pct_du_departement,
    pct_cumul                  AS pct_cumul_dept
FROM v_communes_ranked
WHERE rang <= 10
ORDER BY rang

commune,nb_etablissements,rang,dense_rang,nb_activites_distinctes,cumul_etab,pct_du_departement,pct_cumul_dept
NANTES,74724,1,1,245,74724,55.49,55.5
SAINT-HERBLAIN,10268,2,2,144,84992,7.63,63.1
REZE,7159,3,3,119,92151,5.32,68.4
VERTOU,4819,4,4,104,96970,3.58,72.0
ORVAULT,4742,5,5,93,101712,3.52,75.5
CARQUEFOU,4253,6,6,102,105965,3.16,78.7
SAINT-SEBASTIEN-SUR-LOIRE,3674,7,7,89,109639,2.73,81.4
BOUGUENAIS,3258,8,8,101,112897,2.42,83.8
LA CHAPELLE-SUR-ERDRE,3098,9,9,87,115995,2.3,86.1
COUERON,3040,10,10,105,119035,2.26,88.4


In [0]:
%sql
SELECT
    COALESCE(libelle_secteur, 'Secteur non référencé') AS secteur,
    SUM(nb_etablissements)                           AS total_etab,
    SUM(nb_pme)                                      AS total_pme,
    SUM(nb_ge)                                       AS total_ge,
    COUNT(DISTINCT commune)                          AS nb_communes,
    ROUND(SUM(nb_pme) * 100.0 / SUM(nb_etablissements), 1) AS pct_pme
FROM v_sirene_analytique
GROUP BY libelle_secteur
ORDER BY total_etab DESC
LIMIT 15

secteur,total_etab,total_pme,total_ge,nb_communes,pct_pme
Secteur non référencé,125652,70326,3385,34,56.0
Travaux de construction spécialisés,3234,3204,3,30,99.1
Autres services personnels,923,917,0,26,99.3
Services relatifs aux bâtiments et aménagement paysager,622,608,1,24,97.7
Restauration,526,520,0,26,98.9
Transports terrestres et transport par conduites,465,460,0,25,98.9
Industries alimentaires,395,383,0,24,97.0
Commerce et réparation automobiles,356,349,1,25,98.0
Réparation d'ordinateurs et de biens personnels,308,307,0,25,99.7
Autres industries manufacturières,298,297,0,23,99.7
